# DeepSeek-V4-Flash Cookbook (OpenRouter)

This notebook is a practical starting point for building with DeepSeek-V4-Flash via OpenRouter's OpenAI-compatible API.

What is covered:
- setup and auth
- basic chat completions
- streaming responses
- framework integration (LangChain)
- tool-enabled agent loop
- long-context QA pattern
- structured JSON response pattern
- token usage tracking

Sources:
- https://openrouter.ai/deepseek/deepseek-v4-flash
- https://openrouter.ai/docs

## 0) Setup and Installation

In [ ]:
# Run this once in a fresh environment.
# %pip install -U openai langchain langchain-openai tiktoken

In [ ]:
import os
from openai import OpenAI

MODEL = os.getenv("OPENROUTER_MODEL", "deepseek/deepseek-v4-flash")
BASE_URL = os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1")
API_KEY = os.getenv("OPENROUTER_API_KEY")
SITE_URL = os.getenv("OPENROUTER_SITE_URL", "https://example.com")
APP_NAME = os.getenv("OPENROUTER_APP_NAME", "deepseek-v4-flash-cookbook")

if not API_KEY:
    raise ValueError("Set OPENROUTER_API_KEY before running this notebook.")

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    default_headers={
        "HTTP-Referer": SITE_URL,
        "X-Title": APP_NAME,
    },
)

print(f"Using model: {MODEL}")
print(f"Base URL: {BASE_URL}")
print(f"App: {APP_NAME}")

## 1) Basic Usage with OpenRouter

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise coding assistant."},
        {"role": "user", "content": "List 5 checks to harden a Python API before production launch."},
    ],
)

print(response.choices[0].message.content)

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Explain idempotency keys in one short example."}
    ],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()

## 2) Framework Integration

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    default_headers={
        "HTTP-Referer": SITE_URL,
        "X-Title": APP_NAME,
    },
    temperature=0,
)

msg = llm.invoke("Generate a rollout checklist for introducing a new coding model in CI pipelines.")
print(msg.content)

## 3) Building Agents

In [ ]:
import json

def check_service_dependency(name: str):
    status = {"postgres": "ok", "redis": "ok", "legacy-cache": "deprecated"}
    return {"dependency": name, "status": status.get(name, "unknown")}

tools = [
    {
        "type": "function",
        "function": {
            "name": "check_service_dependency",
            "description": "Return health status for a dependency.",
            "parameters": {
                "type": "object",
                "properties": {"name": {"type": "string"}},
                "required": ["name"],
            },
        },
    }
]

In [ ]:
messages = [
    {"role": "system", "content": "You are a release assistant. Use tools before deciding."},
    {"role": "user", "content": "Can we deploy if the service depends on legacy-cache?"},
]

first = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)

assistant_message = first.choices[0].message
messages.append(assistant_message.model_dump())

if assistant_message.tool_calls:
    for call in assistant_message.tool_calls:
        if call.function.name == "check_service_dependency":
            args = json.loads(call.function.arguments)
            result = check_service_dependency(args["name"])
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": json.dumps(result),
                }
            )

final = client.chat.completions.create(model=MODEL, messages=messages)
print(final.choices[0].message.content)

## 4) Advanced Applications

In [ ]:
def chunk_text(text: str, chunk_size: int = 3000):
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

# Replace with your real long document.
long_doc = """
DeepSeek-V4-Flash supports low-latency production tasks with OpenRouter's unified API.
Use this pattern to feed chunks and ask grounded questions.
""" * 300

chunks = chunk_text(long_doc)
context = "\n\n".join(chunks[:30])

qa = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Answer using only the provided context."},
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: What rollout steps should an engineering team prioritize first?"
        },
    ],
)

print(qa.choices[0].message.content)

In [ ]:
# Structured output pattern without model-specific schema parameters.
# This keeps the example portable across compatible endpoints.
json_resp = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Return valid JSON only."},
        {
            "role": "user",
            "content": "Provide a deploy checklist with fields: risk, owner, mitigation. Return JSON array with 3 items."
        },
    ],
)

payload = json_resp.choices[0].message.content
print(payload)

In [ ]:
usage_demo = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Give 3 short notes on production safety for AI agents."}
    ],
)

usage = usage_demo.usage
print("prompt_tokens:", usage.prompt_tokens)
print("completion_tokens:", usage.completion_tokens)
print("total_tokens:", usage.total_tokens)

## Model-Specific Notes

- OpenRouter model id: deepseek/deepseek-v4-flash.
- OpenAI-compatible endpoint: https://openrouter.ai/api/v1.
- Optional but recommended headers: HTTP-Referer and X-Title for app attribution.
- Check the model card for latest pricing, context limits, and provider routing details.
- Verify feature support (tool calling, streaming, and limits) for your selected route before production rollout.
- Source: https://openrouter.ai/deepseek/deepseek-v4-flash

---
Cookbook done. Replace placeholders with your real prompts, tools, and safety checks, then run each section end-to-end in your target environment.